# 29 carrier lora

Question: does LoRa carry the Homa RPCs of GraphTP inside its budget, in the model of two radios and
then on two boards.

The side under test is the carrier of `aosabi-carrier-lora` over the radio model of `homa::sim`, two
endpoints of the crate `homa` on the compact form, A the client and B the device, each over its
carrier on one radio of a net whose third radio transmits at random, through the link `radio` of
`aosabi-devtool homa-sim`. The reference side is `tests/harness/homa/lora.py`, the time on air of
the SX1261/2 datasheet, revision 1.2, section 6.1.4, the values of a card and the duty cycle over
any continuous hour, written apart from the crate, and this notebook, which knows every byte each
RPC was asked to carry.

Judged stage: H6 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The half this entry runs: model, the model half of stage H6.
half = "model"

# The seed of the radio nets, their losses and interference, and the key.
seed = 20260928

# The calls of each configuration, the loss of each reception per mille, and
# the calls issued at once of the runs back to back.
calls = 1000
loss_permille = 100
saturation_calls = 200

# The share of the DATA budget the calls take, a request and its ACK each: 2
# offers half of what the band carries, so the ledger holds no packet and the
# RESENDs of a lost packet reach the air.
pace_factor = 2

# The restart of the device: the call B holds before it answers, counted
# from 0, how long it holds it, and when B restarts after A issues that call,
# while B holds it and has not answered.
restart_call = 10
restart_hold_ns = 1_000_000_000
restart_after_ns = 500_000_000

pause_sf7_ns = 210_000_000
pause_sf12_ns = 5_100_000_000
pause_slack = 0.05

# The per command timeout, and the one of a cold build.
timeout_s = 600
build_timeout_s = 900

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain, the interface lines of `lora` and `homa-sim`, and the gates the
notebook rests on: the build of the devtool, the lints of the carrier with every feature and of
`homa` with its simulators, and the format of both, every cargo call with `--locked`.

In [ ]:
import json
import os
import random
import statistics
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run
from harness.homa import lora as ref

P = paths(aosabi_root)
OUT = P.out / "29_carrier_lora"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTFMT = CARGO_BIN / "rustfmt"

GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked -p aosabi-carrier-lora --all-features -- -D warnings",
     [CARGO, "clippy", "--locked", "-p", "aosabi-carrier-lora", "--all-features", "--", "-D", "warnings"]),
    ("cargo clippy --locked -p homa --features sim -- -D warnings",
     [CARGO, "clippy", "--locked", "-p", "homa", "--features", "sim", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2021 crates/homa/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2021", P.root / "crates" / "homa" / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-carrier-lora/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-carrier-lora" / "src" / "lib.rs"]),
]
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=build_timeout_s, check=False)
    print("gate %-74s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smokes = [json.loads(run([DEVTOOL, name, "--smoke"], cwd=P.root, timeout_s=timeout_s).out) for name in ("lora", "homa-sim")]
print("repository root:", P.rel(P.root))
print("cargo          :", reported_cargo)
for smoke in smokes:
    print("seam           :", json.dumps(smoke))
assert not GATE_FAILURES, GATE_FAILURES
assert half == "model", half
assert rust_version in reported_cargo, reported_cargo
assert smokes == [{"subcommand": "lora", "implemented": True, "lands_at": "H6"},
                  {"subcommand": "homa-sim", "implemented": True, "lands_at": "H0"}], smokes

PANICS = []
ROWS = []
INFO = []
WAITING = []
DUTY = []
KEY = bytes(random.Random(seed).getrandbits(8) for _ in range(32)).hex()
CONFIGS = [(7, "P"), (9, "P"), (12, "P"), (7, "M"), (9, "M"), (12, "M")]
PAUSE_REQUEST = bytes([0x40, 0x00, 0x00]) + b'{"query":"mutation{pause}"}'
PAUSE_ANSWER = bytes([0x00, 0x00]) + b'{"data":{"pause":true}}'
assert (len(PAUSE_REQUEST), len(PAUSE_ANSWER)) == (30, 25), (len(PAUSE_REQUEST), len(PAUSE_ANSWER))


def row(name, value, bound, ok, mark="MEASURED"):
    """Record one row of the exit number with its measure, its bound and its state."""
    state = "ok" if ok else "CROSSED"
    ROWS.append((name, value, bound, state, mark))
    print("%-64s %-16s %-16s %s" % (name[:64], value, bound, state))


def info(name, value, note, mark="MEASURED"):
    """Record a measure printed as information, with no bound and no assert."""
    INFO.append((name, value, note, mark))
    print("%-64s %-16s %s" % (name[:64], value, note))


def batch(primitive, requests, name):
    """Send requests to one primitive in one process and return its answers."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run([DEVTOOL, primitive, "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False)
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    bad = [a for a in answers if "fatal" in a or "config_error" in a]
    assert not bad, "the seam refused requests of %s: %s" % (name, bad[:2])
    return answers


PROFILES = {}


def card(sf, band, max_rpcs):
    """Return the card's values of a link, from the crate, once for each configuration."""
    key = (sf, band, max_rpcs)
    if key not in PROFILES:
        PROFILES[key] = batch("lora", [{"id": 0, "op": "profile", "sf": sf, "band": band, "max_rpcs": max_rpcs}],
                              "profile-%d-%s-%d" % key)[0]
    return PROFILES[key]


def radio_run(i, sf, band, rpcs, *, max_rpcs=1, window=1, loss=0, interferers=0, baseline=False, **extra):
    """Return the request of one run on the link radio with the card's values of `sf` and `band`."""
    values = card(sf, band, max_rpcs)
    limits = values["limits"]
    timers = values["profile"]["timers"]
    body = {
        "id": i, "op": "run", "link": "radio", "seed": seed + i,
        "profile": {"name": "lora", "sf": sf, "band_permille": ref.DUTY_PERMILLE[band]},
        "overrides": {"unsched_bytes": limits["unsched_bytes"], "window": limits["unsched_bytes"],
                      "timeout_resends": timers["timeout_resends"], "rpc_timeout_ns": timers["rpc_timeout_ns"]},
        "radio": {"link": 1, "key": KEY, "sf": sf, "band": band, "max_rpcs": max_rpcs,
                  "loss_permille": loss, "interferers": interferers, "baseline": baseline},
        "limits_a": limits, "limits_b": limits, "window": window, "rpcs": rpcs,
        "max_time_ns": 2400 * 3600 * 10**9,
    }
    body.update(extra)
    return body


def pause(at_ns=0):
    return {"len": 30, "response_len": 25, "at_ns": at_ns,
            "fill": {"hex": PAUSE_REQUEST.hex()}, "response_fill": {"hex": PAUSE_ANSWER.hex()}}


def duty(answer, band, name):
    """Record the transmissions of A and B of a run for the check of the duty cycle."""
    for radio in (0, 1):
        items = [(x[1], x[2]) for x in answer["radio"]["air"] if x[0] == radio]
        DUTY.append((name, "AB"[radio], band, items))

## The time on air

The crate's `time_on_air` against the formula of `lora.py` for every payload of 1 to 255 bytes at
SF7 to SF12, 125 kHz and the coding rate 4/5, and the reference table of the Homa design note, second
version, section 6.2, printed from the crate. The bench half holds a payload of 64 bytes at SF12
within 2 percent of 2,793.5 ms on the air of two boards, which the model takes from the formula.

In [ ]:
requests = [{"id": 1000 * sf + n, "op": "toa", "sf": sf, "len": n} for sf in range(7, 13) for n in range(1, 256)]
answers = batch("lora", requests, "toa")
toa_differences = sum(1 for q, a in zip(requests, answers) if a["air_ns"] != ref.time_on_air(q["sf"], q["len"]))
AIR = {(q["sf"], q["len"]): a["air_ns"] for q, a in zip(requests, answers)}
print("%-4s %10s %10s %10s %10s %10s %10s" % ("SF", "6 B", "14 B", "26 B", "64 B", "128 B", "255 B"))
for sf in range(12, 6, -1):
    print("%-4s %s" % (sf, " ".join("%10.1f" % (AIR[(sf, n)] / 1e6) for n in (6, 14, 26, 64, 128, 255))))
print("a payload of 64 bytes at SF12:", AIR[(12, 64)] / 1e6, "ms, the design note's 2,793.5 ms")
row("differences of the time on air, 1,530 payloads", toa_differences, "0", toa_differences == 0)

## The values of a card

In [ ]:
value_differences = 0
print("%-8s %-6s %9s %10s %16s %13s %13s %11s" % ("config", "frame", "packet", "longest", "air an hour, s", "resend after", "dead after", "burst gap"))
for sf, band in CONFIGS:
    values = card(sf, band, 1)
    want = (ref.max_frame(band, sf), ref.max_frame(band, sf) - ref.OVERHEAD, ref.longest_response(band, sf))
    got = (values["max_frame"], values["max_packet"], values["longest_response"])
    value_differences += 0 if got == want else 1
    timers = values["profile"]["timers"]
    print("SF%-2d %-3s %6d %9d %10d %16.0f %11.1f s %11.1f s %9.1f ms" % (
        sf, band, got[0], got[1], got[2], values["air_per_hour_ns"] / 1e9,
        timers["resend_after_ns"] / 1e9, timers["rpc_timeout_ns"] / 1e9, values["gap_ns"] / 1e6))
row("differences of the values of a card, six configurations", value_differences, "0", value_differences == 0)

## 1,000 calls

Each configuration, SF7, SF9 and SF12 in the band P and in the band M, runs `calls` pause calls one
at a time, A's LIMITS naming `max_rpcs` 1, with 10 percent of the receptions lost to noise and a
third radio that transmits payloads of 16 to 255 random bytes at random, with no channel activity
detection, for the duty cycle of its band. The calls arrive at the pace `pace_factor` sets: a
request and its ACK of each call take half the DATA share of the band, so the ledger holds no
packet and a RESEND reaches the air when a packet is lost. Section The ledger under calls back to
back runs calls issued at once. A call completes when A holds the whole answer, and a call that ends
otherwise names its failure. The transmissions of A and of B go to the check of the duty cycle.

In [ ]:
requests = []
for i, (sf, band) in enumerate(CONFIGS):
    call_air = ref.time_on_air(sf, 30 + 6 + ref.OVERHEAD) + ref.time_on_air(sf, 10 + ref.OVERHEAD)
    pace = int(pace_factor * call_air * 1000 / (900 * ref.DUTY_PERMILLE[band] / 1000))
    rpcs = [pause(k * pace) for k in range(calls)]
    requests.append(radio_run(i, sf, band, rpcs, loss=loss_permille, interferers=1))
answers = batch("homa-sim", requests, "calls")
least = calls
completed = 0
print("%-8s %9s %12s %9s %10s %10s %10s %9s %9s" % ("config", "complete", "hours", "restarts", "collided", "noise", "busy CAD", "p50, s", "p99, s"))
for (sf, band), answer in zip(CONFIGS, answers):
    done = sum(1 for r in answer["rpcs"] if r["outcome"] == "ok")
    failed = [r["why"] for r in answer["rpcs"] if r["outcome"] != "ok"]
    least = min(least, done)
    completed += done
    waits = sorted((r["response_ns"] - r["issued_ns"]) / 1e9 for r in answer["rpcs"] if r["response_ns"] is not None)
    radio = answer["radio"]
    print("SF%-2d %-3s %9s %12.1f %9d %10d %10d %10d %9.2f %9.2f%s" % (
        sf, band, "%d/%d" % (done, calls), answer["end_ns"] / 3.6e12, sum(r["restarts"] for r in answer["rpcs"]),
        radio["lost"]["collision"], radio["lost"]["noise"], radio["carriers"]["A"]["busy"] + radio["carriers"]["B"]["busy"],
        waits[len(waits) // 2], waits[int(len(waits) * 0.99)], "" if not failed else "  failed: %s" % failed))
    duty(answer, band, "calls SF%d %s" % (sf, band))
row("calls complete in each of the six configurations, the least", "%d of %d" % (least, calls), "%d of %d" % (calls, calls), least == calls)
info("calls complete in all six configurations", "%d of %d" % (completed, calls * len(CONFIGS)), "10 percent loss and a third radio")

## The ledger under calls back to back

The ledger is the flow control of the carrier, and the calls above leave it room. Here
`saturation_calls` calls are issued at once at SF7 in the band P and at SF12 in the band M, with the
same loss and the same third radio, so each end sends as fast as the carrier lets it, and every
transmission of A and of B goes to the check of the duty cycle. How many calls complete is printed
as information, beside the most air in any hour of the busier end against the share of its band,
since the control share of 10 percent then carries the RESENDs of every lost packet. `lora ledger`
then asks one ledger of each band for a script of 20,000 transmissions of drawn length at drawn
times over five hours, two in three of them DATA, as far more than the band carries, and charges
each it admits, and those charges go to the check as well.

In [ ]:
requests = [radio_run(10 + i, sf, band, [pause() for _ in range(saturation_calls)], loss=loss_permille, interferers=1)
            for i, (sf, band) in enumerate([(7, "P"), (12, "M")])]
for (sf, band), answer in zip([(7, "P"), (12, "M")], batch("homa-sim", requests, "full")):
    done = sum(1 for r in answer["rpcs"] if r["outcome"] == "ok")
    failed = {}
    for r in answer["rpcs"]:
        if r["outcome"] != "ok":
            failed[r["why"]] = failed.get(r["why"], 0) + 1
    most = max(ref.Hours([(at, air) for r, at, air, _, _ in answer["radio"]["air"] if r == side]).most()
               for side in (0, 1))
    info("calls back to back complete at SF%d in the band %s" % (sf, band), "%d of %d" % (done, saturation_calls),
         "most air in an hour %.1f s of %d s, failed %s, %.1f h"
         % (most / 1e9, ref.HOUR_NS * ref.DUTY_PERMILLE[band] // 1000 // 10**9, failed or "none", answer["end_ns"] / 3.6e12))
    duty(answer, band, "full SF%d %s" % (sf, band))

scripts = []
for i, band in enumerate(["P", "M"]):
    g = random.Random(seed + 100 + i)
    at = 0
    script = []
    for _ in range(20_000):
        at += g.randrange(0, 2 * 5 * 3600 * 10**9 // 20_000)
        script.append([at, ref.time_on_air(g.choice([7, 9, 12]), g.randint(22, ref.max_frame(band, 12))), g.random() < 1 / 3])
    scripts.append((band, script))
answers = batch("lora", [{"id": i, "op": "ledger", "band": band, "script": script} for i, (band, script) in enumerate(scripts)], "ledger")
for (band, script), answer in zip(scripts, answers):
    charged = [(at, air) for (at, air, control), result in zip(script, answer["results"]) if result["state"] == "charged"]
    data = [(at, air) for (at, air, control), result in zip(script, answer["results"]) if result["state"] == "charged" and not control]
    hours = ref.Hours(charged)
    most = hours.most()
    most_data = ref.Hours(data).most()
    limit = answer["air_per_hour_ns"]
    info("ledger script of the band %s, charged of 20,000" % band, len(charged),
         "most air in an hour %.1f s of %.0f s, DATA %.1f s of %.1f s" % (most / 1e9, limit / 1e9, most_data / 1e9, 0.9 * limit / 1e9))
    DUTY.append(("ledger script %s" % band, "-", band, charged))
    if most_data > 0.9 * limit:
        DUTY.append(("ledger script %s DATA above 90 percent" % band, "-", band, [(0, most_data)]))

## The pause

Each configuration runs one pause on a clean channel and counts the DATA packets each end put on the
air and the air of the two frames. Then each configuration runs the pause behind a long answer, A's
LIMITS naming `max_rpcs` 2: a request whose answer is the longest the link carries, 8,192 bytes at
SF7 to SF9 in the band P and at most 2,048 elsewhere, and the pause sent while that answer is on the
air, after the request and two full frames. The pause must be answered before the long answer
completes. The plan's budget of LoRa, section The budget, lets a small response behind a queued long
answer wait at most one burst, which the exit number of H6 does not name, so its comparison prints
as information: the wait of the pause less the air of its own two frames, beside the longest burst
of the link, 2 s of air or one full frame where that takes longer, and the listening gap after it.

In [ ]:
requests = [radio_run(20 + i, sf, band, [pause()]) for i, (sf, band) in enumerate(CONFIGS)]
one_each_way = 0
PAUSE_AIR = {}
print("%-8s %8s %8s %12s %12s %12s" % ("config", "A DATA", "B DATA", "request, ms", "answer, ms", "both, ms"))
for (sf, band), answer in zip(CONFIGS, batch("homa-sim", requests, "pause")):
    a_data, b_data = answer["links"]["A"]["data"], answer["links"]["B"]["data"]
    one_each_way += 1 if (a_data, b_data) == (1, 1) and answer["rpcs"][0]["outcome"] == "ok" else 0
    first = {radio: next(x for x in answer["radio"]["air"] if x[0] == radio) for radio in (0, 1)}
    both = first[0][2] + first[1][2]
    PAUSE_AIR[(sf, band)] = both
    print("SF%-2d %-3s %8d %8d %12.1f %12.1f %12.1f" % (sf, band, a_data, b_data, first[0][2] / 1e6, first[1][2] / 1e6, both / 1e6))
    if sf in (7, 12):
        bound = pause_sf7_ns if sf == 7 else pause_sf12_ns
        row("air of the pause at SF%d in the band %s, ms" % (sf, band), round(both / 1e6, 1),
            "%.0f +- 5 percent" % (bound / 1e6), abs(both - bound) <= pause_slack * bound)
    duty(answer, band, "pause SF%d %s" % (sf, band))
row("the pause one packet each way, configurations", "%d of %d" % (one_each_way, len(CONFIGS)), "%d of %d" % (len(CONFIGS), len(CONFIGS)),
    one_each_way == len(CONFIGS))

requests = []
LONG = {}
for i, (sf, band) in enumerate(CONFIGS):
    long = ref.longest_response(band, sf) if (band == "P" and sf <= 9) else min(2048, ref.longest_response(band, sf))
    LONG[(sf, band)] = long
    frame = ref.max_frame(band, sf)
    at = ref.time_on_air(sf, 30 + 6 + ref.OVERHEAD) + 2 * ref.time_on_air(sf, frame)
    rpcs = [{"len": 30, "response_len": long, "fill": {"hex": PAUSE_REQUEST.hex()}}, pause(at)]
    requests.append(radio_run(30 + i, sf, band, rpcs, max_rpcs=2, window=2))
BEHIND = {}
before = 0
within = []
print()
print("%-8s %8s %13s %14s %8s %11s %13s %9s %7s" % ("config", "long, B", "long done, s", "pause done, s", "wait, s",
                                                   "own air, s", "beyond it, s", "burst, s", "gap, s"))
for (sf, band), answer in zip(CONFIGS, batch("homa-sim", requests, "behind")):
    long_rpc, pause_rpc = answer["rpcs"]
    ok = long_rpc["outcome"] == pause_rpc["outcome"] == "ok" and pause_rpc["response_ns"] < long_rpc["response_ns"]
    before += 1 if ok else 0
    values = card(sf, band, 2)
    wait = (pause_rpc["response_ns"] - pause_rpc["issued_ns"]) / 1e9
    own = PAUSE_AIR[(sf, band)] / 1e9
    burst = max(values["burst_ns"], values["full_air_ns"]) / 1e9
    gap = values["gap_ns"] / 1e9
    if wait - own <= burst + gap:
        within.append("SF%d %s" % (sf, band))
    print("SF%-2d %-3s %8d %13.2f %14.2f %8.2f %11.2f %13.2f %9.2f %7.2f" % (
        sf, band, LONG[(sf, band)], long_rpc["response_ns"] / 1e9, pause_rpc["response_ns"] / 1e9, wait, own,
        wait - own, burst, gap))
    BEHIND[(sf, band)] = answer
    duty(answer, band, "behind SF%d %s" % (sf, band))
row("the pause answered before the long answer completes, configurations", "%d of %d" % (before, len(CONFIGS)),
    "%d of %d" % (len(CONFIGS), len(CONFIGS)), before == len(CONFIGS))
info("the pause's wait beyond its own air within one burst and its gap", "%d of %d" % (len(within), len(CONFIGS)),
     "the plan's budget of LoRa, no exit number of H6, within: %s" % (", ".join(within) or "none"))

## The restart

Twenty calls at SF7 in the band P at the pace of section 1,000 calls. B holds the request of the
eleventh call, the index `restart_call` counting from 0, for `restart_hold_ns`, 1 s, before it
answers, as a device that takes that long to run an operation, and restarts `restart_after_ns`, 500
ms, after A issues that call, while it holds the request and has not answered it. Its endpoint is
built again with a first identifier of its own, and its carrier restarts as a card does, its
counters after the last reserved block, its window of received counters empty and its held packet
lost. A's RESEND of the call in flight meets `RPC_UNKNOWN`, and A sends the call again, so the row
asks for at least one RPC restarted after `RPC_UNKNOWN` and every call complete. The counter of
every frame on the air, read from its header, must never repeat for one end, since the link, the
direction and the counter are the nonce.

In [ ]:
sf, band = 7, "P"
call_air = ref.time_on_air(sf, 30 + 6 + ref.OVERHEAD) + ref.time_on_air(sf, 10 + ref.OVERHEAD)
pace = int(pace_factor * call_air * 1000 / (900 * ref.DUTY_PERMILLE[band] / 1000))
rpcs = [pause(k * pace) for k in range(20)]
rpcs[restart_call]["hold_ns"] = restart_hold_ns
answer = batch("homa-sim", [radio_run(40, sf, band, rpcs, restart_b_at_ns=restart_call * pace + restart_after_ns)], "restart")[0]
radio = answer["radio"]
reused = 0
for side in ("A", "B"):
    counters = [f[1] for f in radio["frames"][side]]
    reused += len(counters) - len(set(counters))
    print("%s: %d frames, counters %s" % (side, len(counters), counters))
done = sum(1 for r in answer["rpcs"] if r["outcome"] == "ok")
rpc_restarts = sum(r["restarts"] for r in answer["rpcs"])
print("boots:", radio["boots"], " restarted:", answer["restarted_b"], " calls complete:", done, "of 20,",
      "restarts after RPC_UNKNOWN:", rpc_restarts, "of the calls", [i for i, r in enumerate(answer["rpcs"]) if r["restarts"]])
duty(answer, band, "restart SF7 P")
row("restarts of B, RPCs restarted after RPC_UNKNOWN, nonces reused", "%d, %d, %d" % (radio["boots"]["B"] - 1, rpc_restarts, reused),
    "1, at least 1, 0", radio["boots"]["B"] == 2 and answer["restarted_b"] and rpc_restarts >= 1 and reused == 0 and done == 20)

## The rows of GraphTP

Both print as waiting, a visible state and no pass, and H6-RERUN reads this notebook again when they
can run.

In [ ]:
for name, reason in [
    ("1 restarted client whose CANCEL of handle 0 is answered before its first OPERATION runs",
     "waiting until batch G3 of the GraphTP track lands the listener"),
    ("the air of a parked PULL within 5 percent of the design note's section 6.4",
     "waiting until batch G3 with M6 lands the streams over parked PULLs"),
]:
    WAITING.append((name, reason))
    print("waiting: %s, %s, and H6-RERUN runs it" % (name, reason))

## The table against whole messages in sealed frames with no Homa

ADR-0023 asks the plan that owns the carriers to measure GraphTP(Homa) against the frame alone on
each carrier in bytes on the link, latency, throughput, bytes added to each target and resident
memory. On LoRa the frame alone is whole messages in sealed frames with no Homa: each message behind
its length in 4 bytes, cut into packets of the link's size and sealed one a frame by the same
carrier, first in first out, with nothing sent again. Both run the long answer and the pause of
section The pause on a clean channel. The bytes on the link are every frame both ends put on the
air, the latency the wait of the pause, the throughput the long answer's bytes over the time from
its request's arrival to its end, the bytes added the device probe for the HT-CT62, and the resident
memory the storage of a card's endpoint and carrier against the carrier and the buffers of one
message each way.

In [ ]:
requests = []
for i, (sf, band) in enumerate(CONFIGS):
    frame = ref.max_frame(band, sf)
    at = ref.time_on_air(sf, 30 + 6 + ref.OVERHEAD) + 2 * ref.time_on_air(sf, frame)
    rpcs = [{"len": 30, "response_len": LONG[(sf, band)], "fill": {"hex": PAUSE_REQUEST.hex()}}, pause(at)]
    requests.append(radio_run(50 + i, sf, band, rpcs, max_rpcs=2, window=2, baseline=True))
BASE = dict(zip(CONFIGS, batch("homa-sim", requests, "baseline")))

PROBE = P.tests / "fixtures" / "rs" / "homa-probe" / "Cargo.toml"
PROBE_DIR = P.target / "homa-probe-lora"
LLVM_SIZE = __import__("shutil").which("llvm-size")


def probe_bytes(features):
    target_dir = PROBE_DIR / features
    run([CARGO, "build", "--locked", "--release", "--manifest-path", PROBE, "--target", "riscv32imc-unknown-none-elf",
         "--target-dir", target_dir, "--features", features], cwd=P.root, timeout_s=build_timeout_s)
    listing = run([LLVM_SIZE, "-A", target_dir / "riscv32imc-unknown-none-elf" / "release" / "libhoma_probe.a"], timeout_s=120).out
    total, member = 0, False
    for line in listing.splitlines():
        if line.endswith(":") or "(ex " in line:
            member = line.split()[0].startswith("homa_probe")
            continue
        parts = line.split()
        if member and parts and parts[0].split(".")[1:2] and parts[0].startswith((".text", ".rodata", ".srodata")):
            total += int(parts[1])
    return total


frame_only, budget_only, whole = (probe_bytes(f) for f in ("frame-only", "pacer-budget", "sx1262"))
homa_added = whole - frame_only
alone_added = whole - budget_only
sizes = batch("lora", [{"id": 0, "op": "sizes"}], "sizes")[0]
card_config = {"peers": 1, "client_slots": 0, "server_slots": 2, "page_len": 512, "pages": 2 * (8 + 16), "max_gaps": 2}
storage = batch("homa-sim", [{"id": 0, "op": "sizes", "config": card_config, "max_packets": [234]}], "storage")[0]["storage_bytes"]["u8"]
carrier = sizes["carrier"] + sizes["ledger_buckets"]
homa_memory = storage + carrier
alone_memory = carrier + 2 * (8192 + 4)

complete = 0
for sf, band in CONFIGS:
    homa, alone = BEHIND[(sf, band)], BASE[(sf, band)]
    cells = {}
    for name, answer in (("homa", homa), ("alone", alone)):
        frames = answer["radio"]["frames"]
        long_rpc, pause_rpc = answer["rpcs"]
        link_bytes = sum(f[2] for side in ("A", "B") for f in frames[side])
        wait = (pause_rpc["response_ns"] - pause_rpc["issued_ns"]) / 1e6 if pause_rpc["response_ns"] else None
        span = (long_rpc["response_ns"] - long_rpc["request_ns"]) if long_rpc["response_ns"] and long_rpc["request_ns"] else None
        goodput = LONG[(sf, band)] / (span / 1e9) if span else None
        cells[name] = (link_bytes, wait, goodput)
    print("SF%d band %s" % (sf, band))
    rows = [
        ("bytes on the link", cells["homa"][0], cells["alone"][0]),
        ("wait of the pause behind the long answer, ms", cells["homa"][1], cells["alone"][1]),
        ("goodput of the long answer, bytes a second", cells["homa"][2], cells["alone"][2]),
        ("bytes added to the HT-CT62 image", homa_added, alone_added),
        ("resident memory, bytes", homa_memory, alone_memory),
    ]
    for label, a, b in rows:
        print("  %-48s %-14s %-14s" % (label, round(a, 1) if isinstance(a, float) else a, round(b, 1) if isinstance(b, float) else b))
    complete += 1 if all(a is not None and b is not None for _, a, b in rows) else 0
    duty(homa, band, "table homa SF%d %s" % (sf, band))
    duty(alone, band, "table alone SF%d %s" % (sf, band))
print("probe: frame-only %d, pacer-budget %d, sx1262 %d bytes of .text and .rodata" % (frame_only, budget_only, whole))
print("memory: endpoint storage %d, carrier %d with its ledger, one message each way 2 x %d" % (storage, carrier, 8192 + 4))
row("the table against whole messages in sealed frames, configurations complete", "%d of %d" % (complete, len(CONFIGS)),
    "%d of %d" % (len(CONFIGS), len(CONFIGS)), complete == len(CONFIGS))

## The duty cycle

Every transmission of A and of B in every run above, and every charge of the ledger scripts, is held
to the duty cycle of its band: the air of one transmitter inside the hour that ends with each of its
transmissions, counted to the nanosecond over the part of each transmission inside that hour, may
not pass 360 s in the band P and 36 s in the band M. No packet of the band M may take more than 3.6
s of air.

In [ ]:
beyond = 0
longest_m = 0
checked = 0
for name, side, band, items in DUTY:
    limit = ref.HOUR_NS * ref.DUTY_PERMILLE[band] // 1000
    hours = ref.Hours(items)
    over = hours.beyond(limit)
    if name.endswith("DATA above 90 percent"):
        over = items
    beyond += len(over)
    checked += len(items)
    if band == "M":
        longest_m = max([longest_m] + [air for _, air in items])
    if over:
        print("beyond the duty cycle: %s %s, %d transmissions" % (name, side, len(over)))
print("transmissions checked:", checked, " the longest in the band M: %.1f ms" % (longest_m / 1e6))
row("transmissions beyond the duty cycle in any sliding hour", beyond, "0", beyond == 0)
row("packets above 3.6 s in the 1 percent band", sum(1 for _, _, band, items in DUTY if band == "M" for _, air in items if air > 3_600_000_000),
    "0", longest_m <= 3_600_000_000)

## The bench half

In [ ]:
print("not run: the bench half, DC15, two HT-CT62 boards on the bench")

## Verdict

Every row of the exit number prints its measure beside its bound, and a row that crosses its bound
prints CROSSED and fails the notebook. A row that waits prints waiting, a visible state and no pass,
and the verdict names it. A measure printed as information has no bound and no assert.

In [ ]:
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
print()
print("%-64s %-16s %-16s %s" % ("row", "measured", "bound", "state"))
for name, value, bound, state, mark in ROWS:
    print("%-64s %-16s %-16s %s" % (name[:64], value, bound, state))
for name, reason in WAITING:
    print("%-64s %-16s %-16s %s" % (name[:64], "-", "-", "waiting"))
print()
print("information, no bound:")
for name, value, note, mark in INFO:
    print("%-64s %-16s %s, %s" % (name[:64], value, note, mark))
crossed = [name for name, _, _, state, _ in ROWS if state != "ok"]
print()
print("VERDICT: %d of %d rows of the model half hold, %d waiting, %d panics%s" % (
    len(ROWS) - len(crossed), len(ROWS), len(WAITING), len(PANICS), "" if not crossed else ". CROSSED: " + "; ".join(crossed)))
assert not PANICS, PANICS
assert not crossed, crossed